# CS453 — Lab 07: Informed Search
## Greedy Best-First Search and A* · Lecture 5

Misurata University · Faculty of Information Technology · Computer Science Department  
Ali Elzaeilik · Autumn 2026 · Python 3 / Google Colab / Jupyter

Lab 06 used queues and stacks to explore a graph. Informed search uses a heuristic to choose more promising frontier entries. Greedy Best-First Search selects by estimated remaining cost, `h`. A* selects by actual cost so far plus estimated remaining cost, `f = g + h`.

This lab uses the weighted directed graph and heuristic values from revised Lecture 5. UCS is included as a comparison baseline through A* with a zero heuristic. All implementations are iterative and use functions, dictionaries, sets, lists and tuples; no classes or recursion are required.


## 1. Weighted successors and heuristic estimates

A successor entry `(neighbour, step_cost)` records both the next state and the real cost of that directed edge. A loop such as `for neighbour, step_cost in graph[current]` unpacks these two values into separate names.

The lecture's illustrative trip starts at S (Misurata) and ends at G (Tripoli). Edge costs are illustrative litres of fuel, not real travel data. A→D costs 2 but D→A costs 1. C→F and F→C both cost 2. Opposite directions must be stored separately.

`g` is the sum of real edge costs already paid. `h` estimates the remaining cost to G in the same units. `f = g + h` estimates the total cost through a state. The heuristic affects search order; it does not change the graph or the true cost of a returned path.


In [1]:
graph = {
    "S": [("A", 2), ("B", 3), ("C", 4)],
    "A": [("D", 2), ("E", 2)],
    "B": [("E", 1), ("F", 3)],
    "C": [("F", 2), ("H", 3)],
    "D": [("A", 1)],
    "E": [("K", 2)],
    "F": [("C", 2), ("G", 5)],
    "H": [("F", 1), ("M", 2)],
    "K": [("G", 2)],
    "M": [("G", 3)], "G": []
}
heuristic = {"S": 7, "A": 5, "B": 4, "C": 5,
             "D": 6, "E": 4, "F": 3, "H": 3,
             "K": 2, "M": 3, "G": 0}
START = "S"
GOAL = "G"

print("Successors of S:", graph["S"])
print("h(S):", heuristic["S"], "h(G):", heuristic["G"])


Successors of S: [('A', 2), ('B', 3), ('C', 4)]
h(S): 7 h(G): 0


### Priorities after expanding S

| State | g | h | f = g + h |
|---|---|---|---|
| A | 2 | 5 | 7 |
| B | 3 | 4 | 7 |
| C | 4 | 5 | 9 |

Greedy selects B because its h is smallest. A* selects A because A and B tie at f=7 and A was inserted first. UCS also selects A, using g alone.

Successors are generated alphabetically. Among equal priorities, earlier insertion wins. Each pushed entry receives a unique integer insertion number, so a further alphabetical tie is not needed. This reproduces Lecture 5's insertion-first convention.


## 2. A priority queue with heapq

`heapq` provides a priority queue in the Python standard library. `heappush` inserts an entry; `heappop` removes the smallest entry. The underlying list is a heap, not a fully sorted display of the frontier.

Tuples are compared from left to right. In `(priority, insertion_number, state)`, Python first compares priority, then the insertion number. The unique insertion number makes ties deterministic and prevents Python from needing to compare the state values.

`_` is an ordinary variable name conventionally used when a returned value is not needed. In the following example, the two states with priority 7 leave in their insertion order, A before B, even though B is pushed first in the example cell.


In [2]:
from heapq import heappush, heappop
frontier = []
heappush(frontier, (7, 1, "B"))
heappush(frontier, (7, 0, "A"))
heappush(frontier, (9, 2, "C"))
while frontier:
    priority, insertion_number, state = heappop(frontier)
    print(state, priority)


A 7
B 7
C 9


## 3. Path reconstruction and the result record

A parent dictionary records the predecessor of each discovered state. The start has parent `None`. Following parents from the goal produces the path backwards; reversing the list gives the start-to-goal order.

The supplied helpers return one result dictionary. `path` is a connected route, `cost` is the sum of its actual edge costs, and `removed` records the processing order. A failed search has `path=None` and `cost=None`; starting at the goal gives a one-state path with cost 0.

`expanded` counts valid non-goal removals whose successors are inspected. It excludes the successful goal test and stale heap entries. `max_frontier` counts the maximum number of physical waiting heap entries, including stale ones. These are consistent measurements, not estimates of total memory or guarantees about speed.


In [3]:
def reconstruct_path(parent, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path


def path_cost(graph, path):
    if path is None:
        return None
    total = 0
    for index in range(len(path) - 1):
        current = path[index]
        following = path[index + 1]
        found = False
        for neighbour, step_cost in graph.get(current, []):
            if neighbour == following:
                total += step_cost
                found = True
                break
        if not found:
            raise ValueError("path contains an edge not in the graph")
    return total


def search_result(path, cost, removed, expanded, max_frontier, stale_skipped=0):
    return {"path": path, "cost": cost, "removed": removed,
            "expanded": expanded, "max_frontier": max_frontier,
            "stale_skipped": stale_skipped}


def show_result(name, result):
    print(name)
    print("  path:", result["path"])
    print("  real cost:", result["cost"])
    print("  removal order:", result["removed"])
    print("  expanded:", result["expanded"])
    print("  maximum frontier entries:", result["max_frontier"])
    print("  stale entries skipped:", result["stale_skipped"])


## 4. Greedy Best-First Search

Greedy uses only `h` as its priority. It records reached states when scheduling them, as in Lab 06's path functions. Each reached state is scheduled once, with its first parent. On a finite graph this prevents cycling and allows failure to be reported when the frontier is empty.

`cost_so_far` records the real cost of the chosen route so it can be reported, but it does not affect selection. Greedy does not replace an already-discovered route with a cheaper one in this implementation. This is sufficient for finding a path, not for guaranteeing a cheapest path.

`trace=False` is a default parameter. Calling the function with `trace=True` prints processing steps. `sorted(...)` orders successor tuples alphabetically by neighbour name for these letter states.


In [4]:
from heapq import heappush, heappop


def greedy_search(graph, heuristic, start, goal, trace=False):
    frontier = [(heuristic[start], 0, start)]
    reached = {start}
    parent = {start: None}
    cost_so_far = {start: 0}
    order = 0
    removed = []
    expanded = 0
    max_frontier = 1

    while frontier:
        priority, _, current = heappop(frontier)
        removed.append(current)
        if trace:
            print("remove", current, "h =", priority)
        if current == goal:
            path = reconstruct_path(parent, current)
            return search_result(path, cost_so_far[current], removed,
                                 expanded, max_frontier)

        expanded += 1
        for neighbour, step_cost in sorted(graph.get(current, [])):
            if step_cost < 0:
                raise ValueError("this lab requires non-negative edge costs")
            if neighbour not in reached:
                reached.add(neighbour)
                parent[neighbour] = current
                cost_so_far[neighbour] = cost_so_far[current] + step_cost
                order += 1
                heappush(frontier, (heuristic[neighbour], order, neighbour))
        max_frontier = max(max_frontier, len(frontier))

    return search_result(None, None, removed, expanded, max_frontier)


In [5]:
greedy_result = greedy_search(graph, heuristic, START, GOAL, trace=True)
show_result("Greedy Best-First Search", greedy_result)
assert greedy_result["removed"] == ["S", "B", "F", "G"]
assert greedy_result["path"] == ["S", "B", "F", "G"]
assert greedy_result["cost"] == 11
assert path_cost(graph, greedy_result["path"]) == 11


remove S h = 7
remove B h = 4
remove F h = 3
remove G h = 0
Greedy Best-First Search
  path: ['S', 'B', 'F', 'G']
  real cost: 11
  removal order: ['S', 'B', 'F', 'G']
  expanded: 3
  maximum frontier entries: 4
  stale entries skipped: 0


Greedy removes S, B, F, G and returns the route S→B→F→G. It expands three states, since G is goal-tested and returned without expansion. The real route costs 11 litres. The heuristic values are not added into that reported cost.

Greedy can return a cheapest path on some inputs, but this is not guaranteed. Its success on one graph does not establish optimality.


## 5. A*: best known costs and cheaper paths

A* prioritises `f = g + h`. A simple reached set is not enough to reject all later routes: a new route can reach the same state more cheaply. `best_cost[state]` records the cheapest g discovered so far.

A candidate replaces the stored cost and parent only if it is **strictly cheaper**. Equal cost keeps the earlier parent. `best_cost.get(neighbour, float("inf"))` returns the stored cost or positive infinity for an undiscovered state, so a finite first route is accepted.

The heap entry `(f, insertion_number, g, state)` stores both the selection priority f and the actual cost g. A new entry is pushed after an improvement. `heappush` does not edit or remove the old entry.

An entry is stale when its stored g differs from the current best cost. It must be skipped before goal testing or expansion. The goal is accepted when a valid entry is removed, not when the goal is first generated.


In [6]:
def a_star(graph, heuristic, start, goal, trace=False):
    frontier = [(heuristic[start], 0, 0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    order = 0
    removed = []
    expanded = 0
    max_frontier = 1
    stale_skipped = 0

    while frontier:
        priority, _, current_cost, current = heappop(frontier)
        if current_cost != best_cost[current]:
            stale_skipped += 1
            if trace:
                print("skip stale", current, "old g =", current_cost,
                      "best g =", best_cost[current])
            continue

        removed.append(current)
        if trace:
            print("remove", current, "g =", current_cost,
                  "h =", heuristic[current], "f =", priority)
        if current == goal:
            path = reconstruct_path(parent, current)
            return search_result(path, current_cost, removed, expanded,
                                 max_frontier, stale_skipped)

        expanded += 1
        for neighbour, step_cost in sorted(graph.get(current, [])):
            if step_cost < 0:
                raise ValueError("this lab requires non-negative edge costs")
            new_cost = current_cost + step_cost
            if new_cost < best_cost.get(neighbour, float("inf")):
                best_cost[neighbour] = new_cost
                parent[neighbour] = current
                order += 1
                new_priority = new_cost + heuristic[neighbour]
                heappush(frontier, (new_priority, order, new_cost, neighbour))
                if trace:
                    print("  improve", neighbour, "g =", new_cost,
                          "f =", new_priority)
        max_frontier = max(max_frontier, len(frontier))

    return search_result(None, None, removed, expanded, max_frontier, stale_skipped)


In [7]:
a_star_result = a_star(graph, heuristic, START, GOAL, trace=True)
show_result("A*", a_star_result)
assert a_star_result["removed"] == ["S", "A", "B", "E", "K", "G"]
assert a_star_result["path"] == ["S", "A", "E", "K", "G"]
assert a_star_result["cost"] == 8
assert path_cost(graph, a_star_result["path"]) == 8


remove S g = 0 h = 7 f = 7
  improve A g = 2 f = 7
  improve B g = 3 f = 7
  improve C g = 4 f = 9
remove A g = 2 h = 5 f = 7
  improve D g = 4 f = 10
  improve E g = 4 f = 8
remove B g = 3 h = 4 f = 7
  improve F g = 6 f = 9
remove E g = 4 h = 4 f = 8
  improve K g = 6 f = 8
remove K g = 6 h = 2 f = 8
  improve G g = 8 f = 8
remove G g = 8 h = 0 f = 8
A*
  path: ['S', 'A', 'E', 'K', 'G']
  real cost: 8
  removal order: ['S', 'A', 'B', 'E', 'K', 'G']
  expanded: 5
  maximum frontier entries: 4
  stale entries skipped: 0


A* removes S, A, B, E, K, G, expands five states, and returns S→A→E→K→G at cost 8. The equally cheap route through B also exists, but equal-cost candidates do not change parents.

This implementation has no permanent closed-state ban. A strictly cheaper g can schedule a state again, even if it was expanded earlier. This permits reopening when an admissible heuristic is inconsistent. Stale checking handles outdated entries; it is not a substitute for retaining cheaper paths.

For the finite non-negative-cost graphs in this lab, A* has a cheapest-path guarantee with an admissible heuristic, h(goal)=0, goal testing on valid removal, and the cheaper-path/reopening behaviour above. An admissible heuristic never overestimates the true cheapest remaining cost. An arbitrary heuristic does not give that guarantee.


## 6. UCS is A* with h = 0

When every heuristic estimate is zero, `f = g + 0 = g`. The same A* machinery becomes UCS. This baseline lets us isolate what the heuristic changes, without introducing a second priority-queue algorithm.

The wrapper creates a zero value for every graph state and successor, then calls `a_star`. It uses a loop instead of introducing a dictionary comprehension.


In [8]:
def uniform_cost_search(graph, start, goal, trace=False):
    zero_heuristic = {}
    for state, neighbours in graph.items():
        zero_heuristic[state] = 0
        for neighbour, step_cost in neighbours:
            zero_heuristic[neighbour] = 0
    zero_heuristic[start] = 0
    return a_star(graph, zero_heuristic, start, goal, trace)

ucs_result = uniform_cost_search(graph, START, GOAL, trace=True)
show_result("UCS / A* with h=0", ucs_result)
assert ucs_result["cost"] == 8
assert ucs_result["removed"] == ["S", "A", "B", "C", "D", "E", "F", "K", "H", "G"]


remove S g = 0 h = 0 f = 0
  improve A g = 2 f = 2
  improve B g = 3 f = 3
  improve C g = 4 f = 4
remove A g = 2 h = 0 f = 2
  improve D g = 4 f = 4
  improve E g = 4 f = 4
remove B g = 3 h = 0 f = 3
  improve F g = 6 f = 6
remove C g = 4 h = 0 f = 4
  improve H g = 7 f = 7
remove D g = 4 h = 0 f = 4
remove E g = 4 h = 0 f = 4
  improve K g = 6 f = 6
remove F g = 6 h = 0 f = 6
  improve G g = 11 f = 11
remove K g = 6 h = 0 f = 6
  improve G g = 8 f = 8
remove H g = 7 h = 0 f = 7
  improve M g = 9 f = 9
remove G g = 8 h = 0 f = 8
UCS / A* with h=0
  path: ['S', 'A', 'E', 'K', 'G']
  real cost: 8
  removal order: ['S', 'A', 'B', 'C', 'D', 'E', 'F', 'K', 'H', 'G']
  expanded: 9
  maximum frontier entries: 4
  stale entries skipped: 0


## 7. Comparing the strategies

| Measurement | UCS / h=0 | Greedy | A* |
|---|---|---|---|
| Priority | g | h | g+h |
| Returned path | S→A→E→K→G | S→B→F→G | S→A→E→K→G |
| Actual cost | 8 | 11 | 8 |
| Valid removals, including goal | 10 | 4 | 6 |
| Expanded states, excluding goal | 9 | 3 | 5 |
| Maximum physical frontier entries | 4 | 4 | 4 |

On this particular graph, A* saves expansions relative to UCS while retaining the cheapest cost. Greedy expands fewer states here but returns the more expensive route. Frontier size ties here; these examples do not establish a universal performance ranking.


In [9]:
for name, result in [("UCS", ucs_result), ("Greedy", greedy_result),
                     ("A*", a_star_result)]:
    print(name, "cost:", result["cost"],
          "expanded:", result["expanded"],
          "max frontier:", result["max_frontier"])
    assert result["expanded"] == len(result["removed"]) - 1
    assert result["max_frontier"] == 4


UCS cost: 8 expanded: 9 max frontier: 4
Greedy cost: 11 expanded: 3 max frontier: 4
A* cost: 8 expanded: 5 max frontier: 4


## 8. Stale entries and failure tests

A tiny graph makes replacement visible. S→G costs 10, while S→A→G costs 2. With h=0, A is removed first and improves G from 10 to 2. The old G(10) remains in the heap.

Searching for G stops on the valid G(2) entry before G(10) is removed. Searching for absent X forces the finite frontier to drain, exposing the stale G(10) entry. The valid removal list contains each of S, A and G once, and `stale_skipped` is 1.


In [10]:
stale_graph = {"S": [("A", 1), ("G", 10)],
               "A": [("G", 1)], "G": []}
print("Successful search:")
short_result = uniform_cost_search(stale_graph, "S", "G", trace=True)
assert short_result["path"] == ["S", "A", "G"]
assert short_result["cost"] == 2
print("Missing goal, drain the heap:")
stale_result = uniform_cost_search(stale_graph, "S", "X", trace=True)
assert stale_result["path"] is None
assert stale_result["removed"] == ["S", "A", "G"]
assert stale_result["stale_skipped"] == 1
assert a_star(graph, heuristic, "S", "S")["cost"] == 0
assert a_star(graph, heuristic, "S", "X")["path"] is None
assert greedy_search(graph, heuristic, "S", "X")["path"] is None
print("failure, start-at-goal and stale-entry checks passed")


Successful search:
remove S g = 0 h = 0 f = 0
  improve A g = 1 f = 1
  improve G g = 10 f = 10
remove A g = 1 h = 0 f = 1
  improve G g = 2 f = 2
remove G g = 2 h = 0 f = 2
Missing goal, drain the heap:
remove S g = 0 h = 0 f = 0
  improve A g = 1 f = 1
  improve G g = 10 f = 10
remove A g = 1 h = 0 f = 1
  improve G g = 2 f = 2
remove G g = 2 h = 0 f = 2
skip stale G old g = 10 best g = 2
failure, start-at-goal and stale-entry checks passed


## 9. Manhattan distance on the Lab 05 maze

The maze reuses `(row, column)` states and legal actions from Lab 05. A move into a wall or outside the grid is excluded. `transition` rejects illegal actions rather than treating them as stay-in-place moves.

Manhattan distance to the goal is `abs(row - goal_row) + abs(column - goal_column)`. For horizontal/vertical moves of cost 1, it estimates the remaining cost without considering walls. Walls can force extra moves, so the estimate never exceeds the true cheapest number of moves on this grid. A move changes the estimate by at most one, so the heuristic is also consistent here.

Each free cell becomes a graph key with weighted successors `(next_cell, 1)`. The existing A* function then works unchanged. This enumerates only the small teaching maze; larger problems normally generate successors on demand.


In [11]:
maze = ["S.#.", "#.#.", "...G"]
maze_start = (0, 0)
maze_goal = (2, 3)
MOVES = {"Up": (-1, 0), "Down": (1, 0),
         "Left": (0, -1), "Right": (0, 1)}


def actions(state):
    legal = []
    row, column = state
    for action, delta in MOVES.items():
        dr, dc = delta
        nr, nc = row + dr, column + dc
        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]):
            if maze[nr][nc] != "#":
                legal.append(action)
    return legal


def transition(state, action):
    if action not in actions(state):
        raise ValueError("illegal action")
    dr, dc = MOVES[action]
    return (state[0] + dr, state[1] + dc)


def get_successors(state):
    successors = []
    for action in actions(state):
        successors.append(transition(state, action))
    return successors


def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


maze_graph = {}
maze_heuristic = {}
for row in range(len(maze)):
    for column in range(len(maze[0])):
        if maze[row][column] != "#":
            state = (row, column)
            maze_graph[state] = []
            for neighbour in get_successors(state):
                maze_graph[state].append((neighbour, 1))
            maze_heuristic[state] = manhattan(state, maze_goal)

maze_result = a_star(maze_graph, maze_heuristic, maze_start, maze_goal)
show_result("A* on the Lab 05 maze", maze_result)
assert maze_result["path"] == [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
assert maze_result["cost"] == 5


A* on the Lab 05 maze
  path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
  real cost: 5
  removal order: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
  expanded: 5
  maximum frontier entries: 2
  stale entries skipped: 0


## 10. A new starting state

Changing the start to C provides a new problem without changing the goal or heuristic table. A* removes C, F, H, G and returns C→F→G at cost 7. The heuristic still estimates cost to the same goal G.

Greedy also returns cost 7 from C on this graph. This illustrates the distinction between sometimes finding a cheapest path and guaranteeing one in general. If the goal changes, the old heuristic table may no longer estimate cost to the new goal and must be reconsidered.


In [12]:
from_c = a_star(graph, heuristic, "C", "G", trace=True)
show_result("A* from C", from_c)
assert from_c["removed"] == ["C", "F", "H", "G"]
assert from_c["path"] == ["C", "F", "G"]
assert from_c["cost"] == 7
assert greedy_search(graph, heuristic, "C", "G")["cost"] == 7


remove C g = 0 h = 5 f = 5
  improve F g = 2 f = 5
  improve H g = 3 f = 6
remove F g = 2 h = 3 f = 5
  improve G g = 7 f = 7
remove H g = 3 h = 3 f = 6
  improve M g = 5 f = 8
remove G g = 7 h = 0 f = 7
A* from C
  path: ['C', 'F', 'G']
  real cost: 7
  removal order: ['C', 'F', 'H', 'G']
  expanded: 3
  maximum frontier entries: 2
  stale entries skipped: 0


## Summary

Greedy selects by h alone. A* selects by g+h, while reporting only the real edge-cost sum as the path cost. UCS is A* with a zero heuristic. None of these priorities is the number of processed states.

Heap entries need explicit tie-breaking. A* retains strictly cheaper routes, updates parents with best_cost, skips stale entries and tests goals on valid removal. A non-admissible heuristic can remove the cheapest-path guarantee even when the program itself is implemented correctly.

A parent map reconstructs a route, while the removal list records search work. Testing should include reachable and missing goals, start equal to goal, cheaper replacement and stale entries. A heuristic can guide search without changing legal actions or real step costs.
